# Better detection and adversarial fine-tuning

<a href="https://colab.research.google.com/github/fatemehsalmani75/active-subspace-bloodmnist/blob/main/notebooks/robust_training.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The second notebook showed that removing input active directions costs too much accuracy and that the plain
energy score detects the attack only weakly (AUC 0.66). Two follow-ups:

1. **Mahalanobis score.** Measure each direction in units of its natural spread on clean images:
   $s(x) = \sum_j (u_j^T(x-\mu))^2 / \sigma_j^2$.
2. **Adversarial fine-tuning.** Fine-tune the model on images with an AS universal perturbation that is rebuilt on
   the current model every few dozen steps. Control: the same fine-tuning with random noise of the same norm.
   All three models are then attacked again from scratch (adaptive attacks), with 2 different attack sets.

Use a T4 GPU. Total time about 60 min.

In [ ]:
%cd /content
!test -d active-subspace-bloodmnist || git clone -q https://github.com/fatemehsalmani75/active-subspace-bloodmnist.git
%cd /content/active-subspace-bloodmnist
!git pull -q
!pip install -q medmnist adversarial-robustness-toolbox

In [ ]:
import os, copy
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from asnet.data import NUM_CLASSES, get_datasets, make_loader, subset
from asnet.models import build_model
from asnet.utils import set_seed, get_device, train_classifier, stack_dataset
from asnet.attack import as_universal_attack, fooling_rate, predict
from asnet.uap_baseline import uap_baseline
from asnet.defense import (input_active_subspace, detection_scores, mahalanobis_scores,
                           projection_variance, roc_auc)
from asnet.robust import adv_finetune, as_perturbation, noise_perturbation

set_seed(0)
device = get_device()
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
print(device)

## Settings

In [ ]:
SIZE = 64
EPOCHS = 20          # base model
FT_EPOCHS = 5        # fine-tuning
DELTA = 4            # attack / training norm
KS = [5, 10, 20, 50]
N_ATTACK = 200
REFRESH = 60         # rebuild the training perturbation every REFRESH steps
EVAL_SEEDS = [0, 1]

## Base model

In [ ]:
train_aug, _, _ = get_datasets(SIZE, augment=True)
train, val, test = get_datasets(SIZE)
train_loader = make_loader(train_aug, 128, shuffle=True)

base = build_model("vgg19", NUM_CLASSES, SIZE)
if os.path.exists("vgg19_bloodmnist.pt"):
    base.load_state_dict(torch.load("vgg19_bloodmnist.pt", map_location="cpu"))
    base.to(device)
else:
    base, _ = train_classifier(base, train_loader, make_loader(val, 256), epochs=EPOCHS, device=device)
    torch.save(base.state_dict(), "vgg19_bloodmnist.pt")
base.eval()

Xte, yte = stack_dataset(test, device=device)
Xcl, _ = stack_dataset(subset(train, 2000, seed=20), device=device)   # clean images for the statistics
print("base test acc", (predict(base, Xte) == yte).float().mean().item())

## 1. Mahalanobis detection

In [ ]:
sub = input_active_subspace(base, make_loader(subset(train, 2000, seed=10), 64), k=max(KS), device=device)
Xtr, ytr = stack_dataset(subset(train, N_ATTACK, seed=100), device=device)
v_as = as_universal_attack(base, Xtr, ytr, delta=DELTA, log=lambda *a: None).v
v_uap = uap_baseline(base, Xtr, NUM_CLASSES, delta=DELTA, device=device)
g = torch.Generator().manual_seed(0)
v_noise = torch.randn(v_as.shape, generator=g).to(device)
v_noise = v_noise / v_noise.norm() * DELTA

rows = []
for k in KS:
    s = copy.copy(sub); s.U = sub.U[:, :k]
    var = projection_variance(s, Xcl)
    clean_e, clean_m = detection_scores(s, Xte), mahalanobis_scores(s, Xte, var)
    for name, v in [("AS", v_as), ("UAP", v_uap), ("random noise", v_noise)]:
        Xa = (Xte + v).clamp(-1, 1)
        rows.append({"k": k, "attack": name,
                     "energy score": roc_auc(clean_e, detection_scores(s, Xa)),
                     "Mahalanobis": roc_auc(clean_m, mahalanobis_scores(s, Xa, var))})
det = pd.DataFrame(rows)
print("ROC AUC (1.0 = perfect, 0.5 = chance)")
det.pivot(index="k", columns="attack", values=["energy score", "Mahalanobis"]).round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for name, c in [("AS", "C0"), ("UAP", "C1"), ("random noise", "gray")]:
    d = det[det.attack == name]
    ax.plot(d.k, d["Mahalanobis"], "o-", c=c, label=f"{name}, Mahalanobis")
    ax.plot(d.k, d["energy score"], "o:", c=c, alpha=0.6, label=f"{name}, energy")
ax.set(xlabel="k", ylabel="ROC AUC", title=f"detection of universal perturbations, delta = {DELTA}", ylim=(0.4, 1.02))
ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig("detection.png", bbox_inches="tight"); plt.show()

## 2. Fine-tuning

Both fine-tuned models start from the base model and see the same data; only the perturbation differs.

In [ ]:
Xa, ya = stack_dataset(subset(train, 100, seed=300), device=device)   # images used to build training perturbations

set_seed(1)
noise_model = adv_finetune(copy.deepcopy(base), train_loader, noise_perturbation(v_as.shape, DELTA),
                           epochs=FT_EPOCHS, refresh=REFRESH, device=device)
set_seed(1)
as_model = adv_finetune(copy.deepcopy(base), train_loader, as_perturbation(Xa, ya, DELTA),
                        epochs=FT_EPOCHS, refresh=REFRESH, device=device)
torch.save(as_model.state_dict(), "vgg19_bloodmnist_as_trained.pt")

## 3. Attack all three models again

`transfer` = the perturbation built on the base model. `adaptive` = a new attack built on the model being tested,
with attack images that were never used for training.

In [ ]:
models = {"base": base, "noise fine-tuned": noise_model, "AS fine-tuned": as_model}
for m in models.values():
    m.eval()
    for p in m.parameters():
        p.requires_grad_(False)

rows = []
for name, m in models.items():
    clean_acc = (predict(m, Xte) == yte).float().mean().item()
    for seed in EVAL_SEEDS:
        Xs, ys = stack_dataset(subset(train, N_ATTACK, seed=500 + seed), device=device)
        r = {"model": name, "seed": seed, "clean acc": clean_acc,
             "AS transfer": fooling_rate(m, Xte, v_as),
             "UAP transfer": fooling_rate(m, Xte, v_uap),
             "AS adaptive": fooling_rate(m, Xte, as_universal_attack(m, Xs, ys, delta=DELTA, seed=seed,
                                                                     log=lambda *a: None).v),
             "UAP adaptive": fooling_rate(m, Xte, uap_baseline(m, Xs, NUM_CLASSES, delta=DELTA, device=device))}
        rows.append(r)
        print(r)

res = pd.DataFrame(rows)
summary = res.drop(columns="seed").groupby("model", sort=False).agg(["mean", "std"]).round(3)
summary

In [ ]:
cols = ["clean acc", "AS transfer", "UAP transfer", "AS adaptive", "UAP adaptive"]
mean = res.groupby("model", sort=False)[cols].mean() * 100
std = res.groupby("model", sort=False)[cols].std().fillna(0) * 100

fig, ax = plt.subplots(figsize=(11, 4))
x = np.arange(len(cols))
for i, (name, c) in enumerate(zip(mean.index, ["#888", "#c9a227", "#3b6ea5"])):
    ax.bar(x + (i - 1) * 0.27, mean.loc[name], 0.27, yerr=std.loc[name], color=c, label=name, capsize=3)
ax.set_xticks(x, cols)
ax.set(ylabel="%", title=f"clean accuracy and fooling rate (delta = {DELTA}, mean of {len(EVAL_SEEDS)} attack sets)")
ax.legend()
plt.tight_layout(); plt.savefig("robust.png", bbox_inches="tight"); plt.show()

In [ ]:
det.to_csv("detection_mahalanobis.csv", index=False)
res.to_csv("robust_training.csv", index=False)